# 01 — EDA and CX metrics

Phase-1 deliverable: where do bad experiences concentrate (delay, seller, category, state, freight)?

## Conclusions (from the 2026-10-06 run)

- **Modeling window:** 2017-01 → 2018-08 (2016 is a trickle: 4/324/1 orders; 2018-09/10 have just 16/4)
- **Class balance:** 12.8% bad reviews (12,273 / 95,830 delivered+reviewed; avg score 4.16) — imbalanced but workable; PR-AUC and precision@k are the right metrics
- **Where bad experiences concentrate:** delivery delay dominates — bad-review rate climbs from 9.1% (5+ days early) → 12.4% (on time) → 32.1% (1-3 days late) → 67.6% (4-7 late) → 80.1% (8-14 late). Note most deliveries are EARLY (81.6k of 95.8k were 5+ days early).
- **On-time rate by seller size:** flat ~93% across all size buckets, but the smallest sellers (<10 orders, 1,830 sellers) have a much higher bad-review rate (19.6% vs ~14%) — a size/reputation signal, not punctuality

All queries go through DuckDB (`data/olist.duckdb`). Kernel: **Python (cx-risk-engine)**.

In [ ]:
from pathlib import Path
import sys

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from src.db import connect  # noqa: E402

pd.options.display.float_format = '{:,.3f}'.format
con = connect()
print('DuckDB connected:', con.execute("SELECT current_database()").fetchone()[0])

## 1. Row-count reconciliation

In [ ]:
con.execute("""
    SELECT 'raw.orders' AS tbl, COUNT(*) AS n FROM raw.orders
    UNION ALL SELECT 'order_fact', COUNT(*) FROM order_fact
    UNION ALL SELECT 'raw.order_reviews', COUNT(*) FROM raw.order_reviews
    UNION ALL SELECT 'stg_reviews (deduped)', COUNT(*) FROM stg_reviews
    UNION ALL SELECT 'orders with >1 raw review', COUNT(*) FROM
        (SELECT order_id FROM raw.order_reviews GROUP BY order_id HAVING COUNT(*) > 1)
    UNION ALL SELECT 'seller_stats sellers', COUNT(*) FROM seller_stats
    UNION ALL SELECT 'distinct primary sellers in fact', COUNT(DISTINCT primary_seller_id) FROM order_fact
""").df()

## 2. Monthly order volume and coverage

In [ ]:
monthly = con.execute("""
    SELECT DATE_TRUNC('month', order_purchase_timestamp) AS month,
           COUNT(*) AS orders,
           SUM(CASE WHEN is_delivered THEN 1 ELSE 0 END) AS delivered,
           SUM(CASE WHEN review_score IS NOT NULL THEN 1 ELSE 0 END) AS reviewed
    FROM order_fact
    GROUP BY 1 ORDER BY 1
""").df()
monthly['month'] = pd.to_datetime(monthly['month'])
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(monthly['month'], monthly['orders'], label='orders')
ax.bar(monthly['month'], monthly['reviewed'], alpha=0.6, label='reviewed')
ax.set_title('Monthly order volume — coverage is sparse in 2016 and late 2018')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Class balance of the target (`bad_review`, delivered + reviewed)

In [ ]:
con.execute("""
    SELECT
        COUNT(*)                                            AS n_orders,
        SUM(CASE WHEN review_score IS NOT NULL THEN 1 ELSE 0 END) AS n_reviewed,
        SUM(CASE WHEN bad_review = 1 THEN 1 ELSE 0 END)     AS n_bad,
        AVG(CASE WHEN review_score IS NOT NULL
                 THEN bad_review END)                      AS bad_rate_among_reviewed,
        AVG(review_score)                                   AS avg_score
    FROM order_fact
    WHERE is_delivered
""").df()

In [ ]:
con.execute("SELECT review_score, COUNT(*) AS n FROM order_fact WHERE review_score IS NOT NULL GROUP BY 1 ORDER BY 1").df()

## 4. Where do bad experiences concentrate?

All rates below are among **delivered + reviewed** orders.

In [ ]:
BASE = """
    FROM order_fact
    WHERE is_delivered AND review_score IS NOT NULL
"""

def bad_rate(group_expr: str, having: str = "", order: str = "2") -> pd.DataFrame:
    sql = f"""
        SELECT {group_expr} AS grp,
               COUNT(*) AS n,
               AVG(bad_review) AS bad_rate
        {BASE}
        GROUP BY 1 {having}
        ORDER BY {order}
    """
    return con.execute(sql).df()

### 4a. By delivery delay vs promise (the headline driver)

In [ ]:
delay = con.execute(f"""
    SELECT
        CASE
            WHEN delay_days <= -5  THEN 'a. 5+ days early'
            WHEN delay_days <= -1  THEN 'b. 1-4 days early'
            WHEN delay_days = 0    THEN 'c. on time'
            WHEN delay_days <= 3   THEN 'd. 1-3 days late'
            WHEN delay_days <= 7   THEN 'e. 4-7 days late'
            WHEN delay_days <= 14  THEN 'f. 8-14 days late'
            ELSE                        'g. 15+ days late'
        END AS delay_bucket,
        COUNT(*) AS n,
        AVG(bad_review) AS bad_rate,
        AVG(review_score) AS avg_score
    {BASE}
    GROUP BY 1 ORDER BY 1
""").df()
delay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(delay['delay_bucket'], delay['bad_rate'])
ax.set_ylabel('bad-review rate (score <= 2)')
ax.set_title('Bad-review rate by delivery delay vs promise')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 4b. By freight-to-price ratio quartile

In [ ]:
con.execute(f"""
    WITH q AS (
        SELECT bad_review, ntile(4) OVER (ORDER BY freight_ratio) AS fq
        {BASE} AND freight_ratio IS NOT NULL
    )
    SELECT 'Q' || fq AS freight_quartile, COUNT(*) AS n, AVG(bad_review) AS bad_rate
    FROM q GROUP BY 1 ORDER BY 1
""").df()

### 4c. By category (min 200 reviewed orders) — top and bottom 10

In [ ]:
cat = bad_rate("category_en", "HAVING COUNT(*) >= 200 AND category_en IS NOT NULL", "3 DESC, 2 DESC")
display(cat.head(10).rename(columns={'grp': 'category'}))
display(cat.tail(10).rename(columns={'grp': 'category'}))

### 4d. By customer state and seller state

In [ ]:
display(bad_rate("customer_state", order="3 DESC").rename(columns={'grp': 'customer_state'}).head(10))
display(bad_rate("customer_state", order="3").rename(columns={'grp': 'customer_state'}).head(5))

### 4e. By seller (deciles of seller order volume, via seller_stats)

In [ ]:
con.execute("""
    WITH dec AS (
        SELECT ss.bad_review_rate AS rate, ss.n_reviewed_orders,
               ntile(10) OVER (ORDER BY ss.n_orders) AS size_decile
        FROM seller_stats ss
        WHERE ss.n_reviewed_orders >= 10
    )
    SELECT 'D' || size_decile AS seller_size_decile,
           COUNT(*) AS n_sellers,
           AVG(rate) AS avg_bad_review_rate,
           AVG(n_reviewed_orders) AS avg_reviewed_orders
    FROM dec GROUP BY 1 ORDER BY 1
""").df()

## 5. On-time rate by seller size bucket

In [ ]:
con.execute("""
    SELECT
        CASE
            WHEN n_orders < 10    THEN 'a. <10 orders'
            WHEN n_orders < 100   THEN 'b. 10-99'
            WHEN n_orders < 500   THEN 'c. 100-499'
            WHEN n_orders < 2000  THEN 'd. 500-1999'
            ELSE                       'e. 2000+'
        END AS seller_size,
        COUNT(*) AS n_sellers,
        AVG(on_time_rate) AS avg_on_time_rate,
        AVG(late_shipment_rate) AS avg_late_shipment_rate,
        AVG(bad_review_rate) AS avg_bad_review_rate
    FROM seller_stats
    GROUP BY 1 ORDER BY 1
""").df()

## 6. Notes for later phases

- Modeling window chosen here feeds the Phase-2 time-based split.
- `delay_days` / `shipped_late` / `on_time` in `order_fact` are **CX metrics**, NOT model features (leakage rule in `sql/02_order_fact.sql`).
- Seller stats built here are the EDA version; point-in-time seller features come in Phase 2/3.